In [1]:
try:
    import cc3d
    import dijkstra3d
except:
    !pip install dijkstra3d
    !pip install connected-components-3d
import math
import numpy as np
import torch
import sys
sys.path.append('/kaggle/input/vesuvius-surface-marching-ants')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import scipy.ndimage as ndi

In [2]:
!pip install dijkstra3d connected-components-3d matplotlib torch numpy

In [3]:
# #----------------------------------------------------------
# def image_show(name, image, type=None, resize=1):
#     if type == 'rgb2bgr':
#         image = np.ascontiguousarray(image[:, :, ::-1])

#     H, W = image.shape[0:2]
#     cv2.namedWindow(name, cv2.WINDOW_GUI_NORMAL)  # WINDOW_NORMAL
#     # cv2.namedWindow(name, cv2.WINDOW_GUI_EXPANDED)  #WINDOW_GUI_EXPANDED
#     cv2.imshow(name, image)
#     cv2.resizeWindow(name, round(resize * W), round(resize * H))

# def image_norm_show(name, image, type=None, resize=1):
#     image = image.astype(np.float32)
#     if type == 'rgb2bgr':
#         image = np.ascontiguousarray(image[:, :, ::-1])

#     H, W = image.shape[0:2]
#     image = (image-image.min()) / (image.max() - image.min()+1e-8)
#     cv2.namedWindow(name, cv2.WINDOW_GUI_NORMAL)  # WINDOW_NORMAL
#     # cv2.namedWindow(name, cv2.WINDOW_GUI_EXPANDED)  #WINDOW_GUI_EXPANDED
#     cv2.imshow(name, image)
#     cv2.resizeWindow(name, round(resize * W), round(resize * H))


# ########################################3
# ##PARAMETERS
# RAYCAST_IOU=0.1
# SPLIT_RANGE=20
# MIN_POINT_LENGTH=8
# SPLIT_2_MAX_TRIAL=100
# SPLIT_2_REMOVE_THR=0.9
# SPLIT_ALL_MAX_DEPTH=50

# ########################################3
# def add_z(point_yx,z):
#     return np.concatenate([
#         np.full((len(point_yx),1),fill_value=z,dtype=point_yx.dtype),
#         point_yx,
#     ],-1)


# ## ray casting ------------
# def compute_iou_1d(
#     x1, x2
# ):
#     x1 = x1!=0
#     x2 = x2!=0
#     inter = (x1 & x2).sum()
#     union = (x1 | x2).sum()
#     return inter / (union + 1e-6)

# # ray casting test
# def do_ray_casting(
#     point1,  #yx format: Nx2
#     point2,
#     h,w
# ):   
#     # for y
#     by1 = np.bincount(point1[:, 0], minlength=h)
#     by2 = np.bincount(point2[:, 0], minlength=h)
#     #iouy = compute_iou_2d(by1, by2)
#     #print('iouy', iouy)

#     # for x
#     bx1 = np.bincount(point1[:, 1], minlength=w)
#     bx2 = np.bincount(point2[:, 1], minlength=w)
#     #ioux = compute_iou_2d(bx1, bx2)
#     #print('iou', ioux)

#     #we do joint test
#     iou = compute_iou_1d(
#         np.concatenate([by1, bx1]),
#         np.concatenate([by2, bx2]),
#     )
#     return iou

# # test single or multiple surfaces -----
# def split_range(d, min_size=64):
#     """
#     Split range [0, d) into the fewest approximately equal parts,
#     each of size >= min_size.

#     Returns list of (start, end) intervals.
#     """
#     k = max(1, d // min_size)  # number of parts
#     edges = np.linspace(0, d, k + 1, dtype=int)

#     # ensure coverage and monotonicity
#     edges[-1] = d

#     return [(edges[i], edges[i+1]) for i in range(k)]
# '''
# for d in range(1,160,32):
#     print('d', d,  split_range(d, min_size=32))
# exit(0)

# '''


# def find_different_surface_seed(
#     problem,
#     iou_threshold=0.1,
# ):
#     d,h,w = problem.shape
#     split = split_range(d, min_size=SPLIT_RANGE)

#     is_multi_surface = False
#     found=None
#     for z1,z2 in split:
#         if is_multi_surface:
#             break

#         z = (z1+z2)//2 #np.random.randint(z1,z2) # (z1+z2)//2
#         ccz = cc3d.connected_components(problem[z])

#         point = [None]  # bg placeholder
#         for i in range(1, ccz.max() + 1):
#             p = np.stack(np.where(ccz == i)).T
#             if len(p) >= MIN_POINT_LENGTH:
#                 point.append(p)

#         if len(point)<=1:
#             continue

#         #to ray casting test
#         num = len(point)
#         print('found num', num)
#         # found these many blobs
        
#         for i1 in range(1, num):
#             if is_multi_surface:
#                 break

#             for i2 in range(i1+1, num):
#                 iou = do_ray_casting(point[i1], point[i2], h, w)
#                 print('\traycast iou:', i1,i2, iou)
#                 if iou>= iou_threshold:
#                     is_multi_surface = True
#                     found = {
#                         'z' : (z1,z2, z),
#                         'point1_yx' : point[i1],
#                         'point2_yx' : point[i2],
#                     }
#                     break

#     return is_multi_surface, found



# # iteratively  split-----

# def split_problem_to_two(
#     problem,
#     point1_zyx,
#     point2_zyx,
#     is_dilate=True
# ):
#     DEBUG=False
#     sucess  = False
#     problem = problem.copy()

#     for trial in range(SPLIT_2_MAX_TRIAL):
#         path =[]

#         k1 = np.arange(len(point1_zyx))
#         np.random.shuffle(k1)
#         startpoint = point1_zyx[k1[:8]]
#         for (sz, sy, sx) in startpoint:
#             parent = dijkstra3d.parental_field(
#                 np.where(problem, 1.0, 1e6).astype(np.float32), source=(sz, sy, sx), connectivity=26)

#             k2 = np.arange(len(point2_zyx))
#             np.random.shuffle(k2)
#             endppoint =  point2_zyx[k2[:8]]
#             path.extend([
#                 dijkstra3d.path_from_parents(parent, (ez, ey, ex))
#                          for (ez, ey, ex) in endppoint
#             ])
#         path_flat = np.concatenate(path)
#         jump = np.any(~problem[path_flat[:,0],path_flat[:,1],path_flat[:,2]]) #brige in "air"/bg

#         if jump:
#             sucess = True
#             separated =  cc3d.connected_components(problem)
#             if DEBUG: #for debug
#                 show_in_3d(separated, cmap='my_color',title='separated')


#             return sucess, separated
#             #break
#         else:
#             #remove common path and tr
#            # how to get cooment point in path_flat ????


#             uniq, cnt = np.unique(path_flat, axis=0, return_counts=True)
#             order = np.argsort(-cnt)  # descending by count
#             uniq = uniq[order]
#             cnt = cnt[order]
#             print('cnt',trial,  cnt[:5])

#             #for debug
#             if (trial==0) & DEBUG:
#                 overlay3d = problem.astype(np.uint8)
#                 overlay3d[uniq[:,0],uniq[:,1],uniq[:,2]]= 255#cnt+1
#                 show_in_3d(overlay3d, cmap='my_color',title='overlay3d')

#             #remove
#             threshold =SPLIT_2_REMOVE_THR*(cnt.max())
#             u = uniq[cnt>=threshold]
#             if not is_dilate:
#                 problem[u[:, 0], u[:, 1], u[:, 2]] = False  # you can dilate for speedup
#             else:
#                 larger = np.zeros_like(problem, dtype=bool)
#                 larger[u[:, 0], u[:, 1], u[:, 2]]=True
#                 larger = ndi.morphology.binary_dilation(
#                     larger, structure=np.ones((3, 3, 3), dtype=bool), iterations=1)
#                 problem[larger]=False

#     return sucess, problem


# def split_all_surface(problem, result, depth=0, max_depth=SPLIT_ALL_MAX_DEPTH):
#     def collecting_and_return():
#         #show_in_3d(problem, cmap='my_color', title='solved problem')
#         result.append(problem)
#         return
#     #------------------------------

#     print('spliting at depth', depth)
#     if depth >= max_depth:
#         return collecting_and_return()

#     is_multi_surface, found = find_different_surface_seed(
#         problem,
#         iou_threshold=RAYCAST_IOU,
#     )
#     if not is_multi_surface:
#         return  collecting_and_return()

#     print('is_multi_surface', is_multi_surface)
#     print('found:')
#     print('\tz:', found['z'])
#     print('\tpoint1_yx:', len(found['point1_yx']))
#     print('\tpoint2_yx:', len(found['point2_yx']))

#     point1_zyx = add_z(found['point1_yx'], found['z'][2])
#     point2_zyx = add_z(found['point2_yx'], found['z'][2])
    
#     s, solved = split_problem_to_two(
#         problem,
#         point1_zyx,
#         point2_zyx,
#         is_dilate=True
#     )
#     if not s:
#         return  collecting_and_return()

#     split_all_surface(solved==1, result, depth + 1, max_depth)
#     split_all_surface(solved==2, result, depth + 1, max_depth)

In [4]:
#----------------------------------------------------------
def image_show(name, image, type=None, resize=1):
    if type == 'rgb2bgr':
        image = np.ascontiguousarray(image[:, :, ::-1])

    H, W = image.shape[0:2]
    cv2.namedWindow(name, cv2.WINDOW_GUI_NORMAL)  # WINDOW_NORMAL
    # cv2.namedWindow(name, cv2.WINDOW_GUI_EXPANDED)  #WINDOW_GUI_EXPANDED
    cv2.imshow(name, image)
    cv2.resizeWindow(name, round(resize * W), round(resize * H))

def image_norm_show(name, image, type=None, resize=1):
    image = image.astype(np.float32)
    if type == 'rgb2bgr':
        image = np.ascontiguousarray(image[:, :, ::-1])

    H, W = image.shape[0:2]
    image = (image-image.min()) / (image.max() - image.min()+1e-8)
    cv2.namedWindow(name, cv2.WINDOW_GUI_NORMAL)  # WINDOW_NORMAL
    # cv2.namedWindow(name, cv2.WINDOW_GUI_EXPANDED)  #WINDOW_GUI_EXPANDED
    cv2.imshow(name, image)
    cv2.resizeWindow(name, round(resize * W), round(resize * H))


########################################3
##PARAMETERS
RAYCAST_IOU=0.1
SPLIT_RANGE=20
MIN_POINT_LENGTH=8
SPLIT_2_MAX_TRIAL=100
SPLIT_2_REMOVE_THR=0.7 #artfically reducec to remove more chunks 
SPLIT_ALL_MAX_DEPTH=50

########################################3
def add_z(point_yx,z):
    return np.concatenate([
        np.full((len(point_yx),1),fill_value=z,dtype=point_yx.dtype),
        point_yx,
    ],-1)


## ray casting ------------
def compute_iou_1d(
    x1, x2
):
    x1 = x1!=0
    x2 = x2!=0
    inter = (x1 & x2).sum()
    union = (x1 | x2).sum()
    return inter / (union + 1e-6)

# ray casting test
def do_ray_casting(
    point1,  #yx format: Nx2
    point2,
    h,w
):   
    # for y
    by1 = np.bincount(point1[:, 0], minlength=h)
    by2 = np.bincount(point2[:, 0], minlength=h)
    #iouy = compute_iou_2d(by1, by2)
    #print('iouy', iouy)

    # for x
    bx1 = np.bincount(point1[:, 1], minlength=w)
    bx2 = np.bincount(point2[:, 1], minlength=w)
    #ioux = compute_iou_2d(bx1, bx2)
    #print('iou', ioux)

    #we do joint test
    iou = compute_iou_1d(
        np.concatenate([by1, bx1]),
        np.concatenate([by2, bx2]),
    )
    return iou

# test single or multiple surfaces -----
def split_range(d, min_size=64):
    """
    Split range [0, d) into the fewest approximately equal parts,
    each of size >= min_size.

    Returns list of (start, end) intervals.
    """
    k = max(1, d // min_size)  # number of parts
    edges = np.linspace(0, d, k + 1, dtype=int)

    # ensure coverage and monotonicity
    edges[-1] = d

    return [(edges[i], edges[i+1]) for i in range(k)]
'''
for d in range(1,160,32):
    print('d', d,  split_range(d, min_size=32))
exit(0)

'''

"\nfor d in range(1,160,32):\n    print('d', d,  split_range(d, min_size=32))\nexit(0)\n\n"

In [5]:
# ---------------- VERBOSE CONFIG ----------------
VERBOSE = True
VERBOSE_LEVEL = 1
"""
0 = silent
1 = high level steps only
2 = detailed stats
3 = full debug spam
"""

def vlog(level, msg):
    if VERBOSE and level <= VERBOSE_LEVEL:
        print(msg)


In [6]:
def find_different_surface_seed(problem, iou_threshold=0.1):

    d,h,w = problem.shape
    split = split_range(d, min_size=SPLIT_RANGE)

    vlog(1, f"[SeedSearch] Volume shape = {problem.shape}")
    vlog(2, f"[SeedSearch] Z split ranges = {split}")

    is_multi_surface = False
    found=None

    for z1,z2 in split:

        if is_multi_surface:
            break

        z = (z1+z2)//2
        vlog(2, f"\n[SeedSearch] Checking slice z={z} (range {z1}-{z2})")

        ccz = cc3d.connected_components(problem[z])
        num_cc = ccz.max()

        vlog(2, f"[SeedSearch] Connected components found = {num_cc}")

        point = [None]

        for i in range(1, num_cc + 1):
            p = np.stack(np.where(ccz == i)).T

            if len(p) >= MIN_POINT_LENGTH:
                point.append(p)

        vlog(2, f"[SeedSearch] Valid blobs (>={MIN_POINT_LENGTH}) = {len(point)-1}") #earlier u add None, dummmy so -1

        if len(point) <= 1:
            continue

        num = len(point)

        for i1 in range(1, num):

            if is_multi_surface:
                break

            for i2 in range(i1+1, num):

                iou = do_ray_casting(point[i1], point[i2], h, w)

                vlog(3, f"[RayCast] blob {i1} vs {i2} → IOU={iou:.4f}")

                if iou >= iou_threshold:
                    vlog(1, f"✅ [SeedSearch] Multi-surface detected at z={z} (IOU={iou:.3f})")

                    is_multi_surface = True
                    found = {
                        'z' : (z1,z2, z),
                        'point1_yx' : point[i1],
                        'point2_yx' : point[i2],
                    }
                    break

    return is_multi_surface, found

# SPEED CONTROLS
SEED_START_COUNT = 4
SEED_END_COUNT = 4

EARLY_STOP_TRIALS = 5
EARLY_STOP_MIN_REMOVE = -1   # voxels  #early stop is disabled

USE_DOWNSAMPLE_DIJKSTRA = False
DOWNSAMPLE_FACTOR = 2

def split_problem_to_two(problem, point1_zyx, point2_zyx, is_dilate=True, SPLIT_2_REMOVE_THR=0.7):

    vlog(1, "\n[Split2] Starting surface split")
    vlog(2, f"[Split2] Problem voxels = {problem.sum()}")

    success = False
    problem = problem.copy()

    # ⭐ COST MAP CACHE (HUGE SPEED WIN)
    cost_map = np.where(problem, 1.0, 1e6).astype(np.float32)

    # ⭐ Optional downsample
    if USE_DOWNSAMPLE_DIJKSTRA:
        cost_map_ds = cost_map[::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]

    low_remove_counter = 0

    for trial in range(SPLIT_2_MAX_TRIAL):

        vlog(2, f"\n[Split2] Trial {trial+1}/{SPLIT_2_MAX_TRIAL}")

        path = []

        # ⭐ REDUCED SEEDS
        k1 = np.random.permutation(len(point1_zyx))[:SEED_START_COUNT]
        startpoints = point1_zyx[k1]

        for (sz, sy, sx) in startpoints:

            if USE_DOWNSAMPLE_DIJKSTRA:
                parent = dijkstra3d.parental_field(
                    cost_map_ds,
                    source=(sz//DOWNSAMPLE_FACTOR, sy//DOWNSAMPLE_FACTOR, sx//DOWNSAMPLE_FACTOR),
                    connectivity=26
                )
            else:
                parent = dijkstra3d.parental_field(
                    cost_map,
                    source=(sz, sy, sx),
                    connectivity=26
                )

            k2 = np.random.permutation(len(point2_zyx))[:SEED_END_COUNT]
            endpoints = point2_zyx[k2]

            if USE_DOWNSAMPLE_DIJKSTRA:
                paths_local = [
                    dijkstra3d.path_from_parents(
                        parent,
                        (ez//DOWNSAMPLE_FACTOR, ey//DOWNSAMPLE_FACTOR, ex//DOWNSAMPLE_FACTOR)
                    )
                    for (ez, ey, ex) in endpoints
                ]
            else:
                paths_local = [
                    dijkstra3d.path_from_parents(parent, (ez, ey, ex))
                    for (ez, ey, ex) in endpoints
                ]

            path.extend(paths_local)

        if len(path) == 0:
            continue

        path_flat = np.concatenate(path)

        # ⭐ Upscale coords if downsample used
        if USE_DOWNSAMPLE_DIJKSTRA:
            path_flat = path_flat * DOWNSAMPLE_FACTOR

        jump = np.any(~problem[path_flat[:,0], path_flat[:,1], path_flat[:,2]])

        if jump:
            vlog(1, "✅ Natural separation found via air gap")
            success = True
            separated = cc3d.connected_components(problem)
            return success, separated

        # ⭐ PATH VOTING
        uniq, cnt = np.unique(path_flat, axis=0, return_counts=True)

        order = np.argsort(-cnt)
        uniq = uniq[order]
        cnt = cnt[order]

        max_cnt = cnt.max()
        threshold = SPLIT_2_REMOVE_THR * max_cnt
        u = uniq[cnt >= threshold]

        vlog(2, f"[Split2] Removing {len(u)} voxels (thr={threshold:.2f})")

        # ⭐ EARLY STOP HEURISTIC
        if len(u) < EARLY_STOP_MIN_REMOVE:
            low_remove_counter += 1
        else:
            low_remove_counter = 0

        if low_remove_counter >= EARLY_STOP_TRIALS:
            vlog(1, "⚠ Early stop: low removal repeatedly")
            break

        if not is_dilate:
            problem[u[:,0], u[:,1], u[:,2]] = False
        else:
            larger = np.zeros_like(problem, dtype=bool)
            larger[u[:,0], u[:,1], u[:,2]] = True

            larger = ndi.binary_dilation(
                larger,
                structure=np.ones((3,3,3), dtype=bool),
                iterations=1
            )

            problem[larger] = False

         # ⭐ Check if split achieved

        #Checking right after removal
        
        cc = cc3d.connected_components(problem)
        
        if cc.max() >= 2:
            vlog(1, f"✅💡 Split achieved after removal (CC={cc.max()})")
            success = True
            return success, cc

        # ⭐ Update cost map after removal
        cost_map = np.where(problem, 1.0, 1e6).astype(np.float32)

        if USE_DOWNSAMPLE_DIJKSTRA:
            cost_map_ds = cost_map[::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR, ::DOWNSAMPLE_FACTOR]


    vlog(1, "❌ Failed to split after trials")
    return success, problem


def split_all_surface(problem, result, depth=0, max_depth=SPLIT_ALL_MAX_DEPTH, SPLIT_2_REMOVE_THR=0.9):

    indent = "  " * depth

    vlog(1, f"\n{indent}[SplitAll] Depth={depth}")
    vlog(2, f"{indent}[SplitAll] Voxels={problem.sum()}")

    if depth >= max_depth:
        vlog(1, f"{indent} Max depth reached")
        result.append(problem)
        return

    is_multi_surface, found = find_different_surface_seed(
        problem,
        iou_threshold=RAYCAST_IOU,
    )

    if not is_multi_surface:
        vlog(1, f"{indent}✅ Single surface — storing result")
        result.append(problem)
        return

    vlog(1, f"{indent}🔥 Multi surface found")
    vlog(2, f"{indent}Seed Z = {found['z']}")
    vlog(2, f"{indent}Blob sizes = {len(found['point1_yx'])}, {len(found['point2_yx'])}")

    point1_zyx = add_z(found['point1_yx'], found['z'][2])
    point2_zyx = add_z(found['point2_yx'], found['z'][2])

    s, solved = split_problem_to_two(
        problem,
        point1_zyx,
        point2_zyx,
        is_dilate=True,
        SPLIT_2_REMOVE_THR=SPLIT_2_REMOVE_THR
    )

    if not s:
        vlog(1, f"{indent}❌ Split failed — storing original")
        result.append(problem)
        return

    split_all_surface(solved==1, result, depth+1, max_depth)
    split_all_surface(solved==2, result, depth+1, max_depth)


In [7]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def show_in_3d(data, cmap="tab20", title=None, elev=30, azim=120):
    vol = data[::2,::2,::2] # already thresholded and labelled 
    #too slow so we subsmple
    
    # boolean mask of voxels to draw
    #vol = vol[:10]
    max_label = vol.max()

    # auto generate colors
    color = plt.cm.tab20(np.linspace(0, 1, max_label+1))

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")

    facecolors = np.zeros(vol.shape + (4,), dtype=float)
    for label in range(1,max_label+1):
        filled = vol==label
        facecolors[filled, :4] = color[label-1]    # RGBA tuple
        facecolors[filled,  3] = 0.6

    
    ax.voxels(vol!=0, facecolors=facecolors, edgecolor='none')
    ax.view_init(elev=elev, azim=azim)
    ax.set_box_aspect([s for s in vol.shape])  # equal aspect
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    plt.tight_layout()
    plt.show()

In [8]:
import numpy as np
import cc3d

def ant_walk_inference(
    binary_mask,
    min_cc_size=500,
    max_depth=50,
    SPLIT_2_REMOVE_THR=0.7,
):
    """
    Final ant-walk based surface separation for inference.

    Parameters
    ----------
    binary_mask : np.ndarray (Z,Y,X)
        Binary mask from topo_postprocess (0/1).
    min_cc_size : int
        Skip connected components smaller than this.
    max_depth : int
        Maximum recursive split depth.

    Returns
    -------
    final_labels : np.ndarray (Z,Y,X), uint16
        0 = background
        1..N = separated surfaces
    """

    binary_mask = binary_mask.astype(bool)

    # 1) Connected components
    cc = cc3d.connected_components(binary_mask)
    stats = cc3d.statistics(cc)

    final_labels = np.zeros_like(cc, dtype=np.uint16)
    next_label = 1

    # 2) Process each CC independently
    for cc_id in range(1, cc.max() + 1):

        voxel_count = stats["voxel_counts"][cc_id]
        if voxel_count < min_cc_size:
            continue  # skip noise

        (zs, ys, xs) = stats["bounding_boxes"][cc_id]
        z1, z2 = zs.start, zs.stop
        y1, y2 = ys.start, ys.stop
        x1, x2 = xs.start, xs.stop

        problem = (cc == cc_id)[z1:z2, y1:y2, x1:x2]

        # 3) Recursive ant-walk splitting
        pieces = []
        split_all_surface(
            problem,
            pieces,
            depth=0,
            max_depth=max_depth,
            SPLIT_2_REMOVE_THR=SPLIT_2_REMOVE_THR
        )

        # 4) Write back results
        for p in pieces:
            final_labels[z1:z2, y1:y2, x1:x2][p] = next_label
            next_label += 1

        print(f"DONE FOR CC_ID: {cc_id}")

    return final_labels


In [9]:


# #example usage
# def do_demo(
#     id, problem_id
# ):

#     #id = 63469416 #11630450
#     npz = np.load(f'{dump_dir}/{id}-out.npz')
#     prob = npz['arr_0']   #unet voxel label results
#     trurth = npz['arr_1'] #ground truth
#     ignore = npz['arr_2'] #unlabelled region: true/false
#     print('prob', prob.shape)
    
#     prob1= prob.copy()
#     prob1[ignore]=0  #for simplicity we ignore unlabelled region

    
#     thresh1 = prob1>0.3
#     cc1 = cc3d.connected_components(thresh1)
#     stats1 = cc3d.statistics(cc1)
#     print('stats1', stats1['voxel_counts'])
    
#     print('cc1.max()', cc1.max())
#     #show_in_3d(cc1,cmap='my_color',title='cc1')
    
#     #############################################################
#     i = problem_id# specific problem
    
#     (zs, ys, xs) = stats1['bounding_boxes'][i]
#     z1, z2 = zs.start, zs.stop
#     y1, y2 = ys.start, ys.stop
#     x1, x2 = xs.start, xs.stop
#     print('z1, z2', z1, z2)
    
#     problem = (cc1==i)[z1:z2, y1:y2, x1:x2]
#     show_in_3d(problem,cmap='my_color',title='problem')
    
 
#     solved = []
#     split_all_surface(problem, solved, depth=0, max_depth=50)

#     final_ant = ant_walk_inference(problem)

#     # print('')
#     # print('****************************************************')
#     # print('split_all_surface', len(solved))
    
#     # overlay3d=np.zeros_like(problem, dtype=np.uint8)
#     # for i,s in enumerate(solved):
#     #     print('adding:',i, s.sum())
#     #     overlay3d[s!=0]=i+1

#     #     #show one by one
#     #     #show_in_3d(s.astype(np.uint8)*(i+1), cmap='my_color', title='s')
    
#     # #show all
#     # show_in_3d(overlay3d, cmap='my_color', title='solved', elev=30, azim=120)
#     # show_in_3d(overlay3d, cmap='my_color', title='solved', elev=60, azim=120)


# #results are from a better-trained UNET model: 
# #   https://www.kaggle.com/code/hengck23/demo-limit-of-good-unet-pixel-predict

# #this is a simple case where 3 surfaces are stuck together
#     return final_ant

# dump_dir = "/kaggle/input/vesuvius-surface-marching-ants/dump"

# id = 11630450
# problem_i =4

# ginale = do_demo(id, problem_i)
# show_in_3d(ginale)

In [10]:
# import time

# time1 = time.time()

# pred_map = np.load("/kaggle/input/pred-map/kaggle/working/val_preds.npy")
# #GOTTA SQUEEZE PRED_MAP
# pred_map = pred_map.squeeze(0)
# final_map = ant_walk_inference(pred_map, SPLIT_2_REMOVE_THR=0.5) #0.5 gives result very fast

# time2 = time.time()

# print(f"It TOOK around: {time2-time1} secs")
# #Apply post process on it okay -> 

In [11]:
# np.save("final.npy", final_map)

In [12]:
# import os

# def print_all_files(path="."):
#     for root, dirs, files in os.walk(path):
#         for file in files:
#             print(os.path.join(root, file))

# # Usage
# print_all_files("/kaggle/input/nnunet-preds-5/kaggle/working")


In [13]:
# !zip gg.zip /kaggle/working/final.npy

In [14]:
import time
import os

for path in ["1059332280_pred.npy", "1029212680_pred.npy", "1033784946_pred.npy", "1004283650_pred.npy", "1061356924_pred.npy", "102536988_pred.npy"]:

    time1 = time.time()

    g_path = os.path.join("/kaggle/input/nnunet-preds-5/kaggle/working", path)
    
    pred_map = np.load(g_path)
    #GOTTA SQUEEZE PRED_MAP
    # pred_map = pred_map.squeeze(0)
    final_map = ant_walk_inference(pred_map, SPLIT_2_REMOVE_THR=0.9) #0.5 gives result very fast

    #you convert the lables into 1s and 0s
    final_map = (final_map > 0).astype(np.int8)

    np.save("post_"+path, final_map)
    
    time2 = time.time()
    
    print(f"It TOOK around: {time2-time1} secs")
#Apply post process on it okay -> 


[SplitAll] Depth=0
[SeedSearch] Volume shape = (49, 76, 12)
✅ Single surface — storing result
DONE FOR CC_ID: 1

[SplitAll] Depth=0
[SeedSearch] Volume shape = (77, 235, 29)
✅ Single surface — storing result
DONE FOR CC_ID: 2

[SplitAll] Depth=0
[SeedSearch] Volume shape = (89, 314, 68)
✅ Single surface — storing result
DONE FOR CC_ID: 3

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 104)
✅ Single surface — storing result
DONE FOR CC_ID: 4

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 123)
✅ Single surface — storing result
DONE FOR CC_ID: 5

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 143)
✅ Single surface — storing result
DONE FOR CC_ID: 6

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 134)
✅ Single surface — storing result
DONE FOR CC_ID: 7

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 108)
✅ Single surface — storing result
DONE FOR CC_ID: 8

[SplitAll] Depth=0
[SeedSearch] Volume shape = (318, 314, 126)
✅ Single surfac

In [15]:

!zip final_6_th_is_0_dot_5.zip post_*.npy

  adding: post_1004283650_pred.npy (deflated 97%)
  adding: post_102536988_pred.npy (deflated 96%)
  adding: post_1029212680_pred.npy (deflated 97%)
  adding: post_1033784946_pred.npy (deflated 97%)
  adding: post_1059332280_pred.npy (deflated 97%)
  adding: post_1061356924_pred.npy (deflated 98%)


In [16]:
# #Visualzie
# %matplotlib inline
# import numpy as np
# import tifffile
# import matplotlib.pyplot as plt
# import ipywidgets as widgets
# from IPython.display import display

# def compare_3d_volume(
#   vol1,
#   vol2,
#   axis=0,
#   cmap="viridis"
# ):
#   # vol1 = tifffile.imread(tiff_path_1)
#   # vol2 = tifffile.imread(tiff_path_2)

#   assert vol1.shape == vol2.shape
#   assert axis in (0, 1, 2)

#   vmin = min(vol1.min(), vol2.min())
#   vmax = max(vol1.max(), vol2.max())

#   def get_slice(vol, idx):
#     if axis == 0:
#       return vol[idx]
#     elif axis == 1:
#       return vol[:, idx, :]
#     else:
#       return vol[:, :, idx]

#   slider = widgets.IntSlider(
#     min=0,
#     max=vol1.shape[axis] - 1,
#     step=1,
#     value=vol1.shape[axis] // 2,
#     description="Slice",
#     continuous_update=False
#   )

#   out = widgets.Output()

#   def update(change=None):
#     idx = slider.value
#     with out:
#       out.clear_output(wait=True)
#       fig, axes = plt.subplots(1, 2, figsize=(12, 6))

#       # REMOVE vmin=vmin, vmax=vmax to let matplotlib auto-scale
#       im1 = axes[0].imshow(get_slice(vol1, idx), cmap=cmap)
#       axes[0].set_title(f"Vol 1 (Auto-scaled) | Slice {idx}")
#       plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
#       axes[0].axis("off")

#       im2 = axes[1].imshow(get_slice(vol2, idx), cmap=cmap)
#       axes[1].set_title(f"Vol 2 (Auto-scaled) | Slice {idx}")
#       plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
#       axes[1].axis("off")

#       plt.show()

#       plt.show()

#   slider.observe(update, names="value")

#   display(widgets.VBox([slider, out]))
#   update()

    

In [17]:
# final_map = (final_map > 0).astype(np.uint8)

# compare_3d_volume(pred_map, final_map)

In [18]:
# original = tifffile.imread("/kaggle/input/vesuvius-challenge-surface-detection/train_images/1004283650.tif")

# compare_3d_volume(original, final_map)

In [19]:
# import numpy as np

# def make_two_sheets_two_straight_bridges(
#     Z=80,
#     Y=160,
#     X=160,
#     sheet_thickness=3,
#     bridge_width=4
# ):
#     vol = np.zeros((Z, Y, X), dtype=bool)

#     # -------------------
#     # SHEET A
#     # -------------------
#     zA = 15
#     vol[
#         zA : zA + sheet_thickness,
#         30:110,
#         20:140
#     ] = True

#     # -------------------
#     # SHEET B
#     # -------------------
#     zB = 55
#     vol[
#         zB : zB + sheet_thickness,
#         35:115,
#         25:145
#     ] = True

#     return vol

# sheets = make_two_sheets_two_straight_bridges()

In [20]:
# show_in_3d(sheets, azim=60)

In [21]:
# finale = ant_walk_inference(sheets)

In [22]:
# show_in_3d(finale, azim=60)

In [23]:
# np.save("final.npy", final_map)a

In [24]:
print("HI")

HI


In [25]:
print("HI")

HI


In [26]:
print("HI")

HI
